# FNO Heston Pricing Engine - Training on Colab T4 GPU
This notebook trains the Fourier Neural Operator surrogate model directly on the attached Google Colab NVIDIA T4 GPU.

In [ ]:
# Step 1: Verify CUDA & GPU
import torch

print("PyTorch version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    print(f"Using GPU: {device_name}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    raise RuntimeError("No GPU detected! In VS Code, select 'New Colab Server' -> 'Colab GPU T4'.")

In [ ]:
# Step 2: Clone repository & install dependencies
import os

REPO_PATH = "/content/FNO-for-Heston-Pricing-Engine-"
if not os.path.exists(REPO_PATH):
    print("Cloning repository...")
    !git clone https://github.com/bananapants-87/FNO-for-Heston-Pricing-Engine-.git {REPO_PATH}
else:
    print("Repository exists. Pulling latest code...")
    !cd {REPO_PATH} && git pull

os.chdir(REPO_PATH)
print("Active working directory:", os.getcwd())

print("\nInstalling neuraloperator and heston-fno package...")
!pip install -q neuraloperator pyyaml
!pip install -q -e .
print("Environment setup complete!")

In [ ]:
# Step 3: Organize Data Files & Connect targets.pt
import os
import shutil
from pathlib import Path

DATA_DIR = Path("/content/FNO-for-Heston-Pricing-Engine-/data/final")
DATA_DIR.mkdir(parents=True, exist_ok=True)

# 1. Automatically collect files already uploaded to /content/final or /content
search_locations = [Path("/content/final"), Path("/content/data/final"), Path("/content")]
for loc in search_locations:
    if loc.exists() and loc.resolve() != DATA_DIR.resolve():
        for fname in ["inputs.pt", "s_grid.pt", "v_grid.pt", "targets.pt"]:
            src_f = loc / fname
            dst_f = DATA_DIR / fname
            if src_f.exists() and not dst_f.exists():
                shutil.copy2(str(src_f), str(dst_f))
                print(f"Copied {fname} from {loc} -> {DATA_DIR}")

# 2. If targets.pt is in Google Drive, copy it over
drive_candidates = [
    Path("/content/drive/MyDrive/targets.pt"),
    Path("/content/drive/MyDrive/final/targets.pt"),
]
target_f = DATA_DIR / "targets.pt"
if not target_f.exists():
    for dc in drive_candidates:
        if dc.exists():
            print(f"Found targets.pt in Google Drive: {dc}")
            shutil.copy2(str(dc), str(target_f))
            print(f"Successfully imported targets.pt ({target_f.stat().st_size / (1024*1024):.2f} MB)!")
            break

# 3. Verify all required files
required = ["inputs.pt", "targets.pt", "s_grid.pt", "v_grid.pt"]
all_present = True
print("\n--- Dataset File Status ---")
for f in required:
    p = DATA_DIR / f
    if p.exists():
        print(f"  {f:<12}: FOUND ({p.stat().st_size / (1024*1024):.2f} MB)")
    else:
        print(f"  {f:<12}: MISSING")
        all_present = False

if all_present:
    print("\nAll 4 dataset files are verified! You can proceed to Step 4 & Step 5.")
else:
    print("\n>>> targets.pt is ~390 MB and cannot be uploaded via the VS Code extension limit.")
    print(">>> Quickest 1-minute fix using Google Drive:")
    print("    1. Open https://drive.google.com and drag 'targets.pt' into your Google Drive.")
    print("    2. Run the 'Mount Google Drive' cell below.")
    print("    3. Re-run this cell.")

In [ ]:
# [Optional] Mount Google Drive to import targets.pt
# Run this cell if you uploaded targets.pt to your Google Drive
from google.colab import drive
drive.mount('/content/drive')
print("Google Drive mounted at /content/drive")

In [ ]:
# Step 4: GPU Smoke Test - verify FNO model compiles and runs on T4 GPU
import torch
import yaml
from heston_fno.models.fno import build_model_from_config

with open("configs/fno.yaml", "r") as fh:
    cfg = yaml.safe_load(fh)

model = build_model_from_config(cfg).cuda()
sample_x = torch.randn(2, 7, 32, 32, device="cuda")
sample_y = model(sample_x)

print("Smoke Test Succeeded!")
print("  Model Device:", next(model.parameters()).device)
print("  Sample Output Shape:", tuple(sample_y.shape))
print("  Output Device:", sample_y.device)

In [ ]:
# Step 5: Run Full Training on NVIDIA T4 GPU
!python scripts/train_fno.py